In [1]:
import pandas as pd
import re
from datetime import datetime
import numpy as np
from plotnine import ggplot, geom_bar, aes, geom_col, labs, theme, element_blank, element_text, geom_point, geom_histogram, facet_wrap, geom_line, scale_y_log10, labeller
import os

In [2]:
# Define [current] NCBI and BVBRC metadata

# UPDTNCBI = "./metadata/sequence_metadata/NCBIdata_08132024_updated_deduped.tsv"
# UPDTBVBRC = "./metadata/sequence_metadata/BVBRCdata_08132024_updated_deduped.tsv"
# UPDTGIS = "./metadata/sequence_metadata/GISAIDdata_09182024_expanded_deduped.tsv"

# UPDTNCBI = "./metadata/sequence_metadata/ncbi_2022-2024_updated_deduped.tsv"
# UPDTBVBRC = "./metadata/sequence_metadata/bvbrc_2022-2024_updated_deduped.tsv"
# UPDTGIS = "./metadata/sequence_metadata/gisaid_expanded_deduped_dwnld-2025-01-09.tsv"

UPDTNCBI = "./metadata/sequence_metadata/ncbi_nov2021-2024_updated_deduped.tsv"
UPDTBVBRC = "./metadata/sequence_metadata/bvbrc_nov2021-2024_updated_deduped.tsv"
UPDTGIS = "./metadata/sequence_metadata/gisaid_expanded_deduped_dwnld-2025-05-25.tsv"

In [3]:
# Import NCBI and BVBRC metadata

data_ncbi = pd.read_csv(UPDTNCBI, sep = "\t")
data_bvbrc = pd.read_csv(UPDTBVBRC, sep = "\t")
data_gisaid = pd.read_csv(UPDTGIS, sep = "\t")

/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_61953/440696993.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_61953/440696993.py:4: DtypeWarning: Columns (13,15,30) have mixed types. Specify dtype option on import or set low_memory=False.


In [4]:
# Lengths of the datasets

print(len(data_ncbi))

print(len(data_bvbrc))

print(len(data_gisaid))

# NCBI: 13965, BVBRC: 14424, GISAID: 44609
# NCBI: 36489, BVBRC: 14984, GISAID: 65071
# NCBI: 40089, BVBRC: 39279, GISAID: 75704

40089
39279
75704


In [5]:
# Make a new date column with non-ambiguous dates for the GISAID metadata
# This code (courtesy of Chat GPT) creates a new dataframe with a "Collection_datetime" object that has reformatted dates

data_gisaid_nonambigdates = data_gisaid.copy()

data_gisaid_nonambigdates['Collection_datetime'] = None

for idx, i in enumerate(data_gisaid_nonambigdates["date"]):
    # Check if the date format is '%Y-%m-XX' (e.g., '2023-08-XX')
    try:
        if datetime.strptime(i, '%Y-%m-XX'):
            # Convert the date to the format '%Y-%m-15' (e.g., '2023-01-15')
            formatted_date = datetime.strptime(i, '%Y-%m-XX').strftime('%Y-%m-15')
            # Update the DataFrame with the formatted date
            data_gisaid_nonambigdates.at[idx, 'Collection_datetime'] = formatted_date
    except ValueError:
        pass

    # Check if the date format is ''%Y-%m-%d'
    try:
        if datetime.strptime(i, '%Y-%m-%d'):
            formatted_date = datetime.strptime(i, '%Y-%m-%d').strftime('%Y-%m-%d')
            data_gisaid_nonambigdates.at[idx, 'Collection_datetime'] = formatted_date
    except ValueError:
        pass

In [6]:
# Check that the length of the GISAID metadata is the same

print(len(data_gisaid_nonambigdates))

75704


In [7]:
# Merge NCBI and BVBRC

ncbibvbrc_mergedmeta = pd.merge(data_ncbi, data_bvbrc, how = "outer", on = "strain_segment", indicator = "_merge_1")

In [8]:
# Make an indicator column for Merge #1

print(ncbibvbrc_mergedmeta["_merge_1"].value_counts())

merge_1_dict = {"both":"ncbibvbrc", "left_only":"ncbi", "right_only":"bvbrc"}

ncbibvbrc_mergedmeta["source"] = ncbibvbrc_mergedmeta["_merge_1"].map(merge_1_dict)

# August/September 2024 pull: Both: 12736, NCBI: 1229, BVBRC: 1688
# January 2025 pull: 13365 both, 23124 NCBI, 1619 BVBRC
# May 2025 pull: 37508 both, 2581 NCBI, 1771 BVBRC

_merge_1
both          37508
left_only      2581
right_only     1771
Name: count, dtype: int64


In [9]:
# Merge in GISAID

ncbibvbrcgis_mergedmeta = pd.merge(ncbibvbrc_mergedmeta, data_gisaid_nonambigdates, how = "outer", on = "strain_segment", indicator = "_merge_2")

In [10]:
# Make an indicator column for Merge #2

ncbibvbrcgis_mergedmeta["source"] = ncbibvbrcgis_mergedmeta["source"].replace(" ", "nan")

ncbibvbrcgis_mergedmeta["merge1_merge2"] = ncbibvbrcgis_mergedmeta[["source", "_merge_2"]].astype(str).apply("_".join, axis=1)

print(ncbibvbrcgis_mergedmeta["merge1_merge2"].value_counts())

merge_2_dict = {"nan_right_only":"gisaid",
                "ncbibvbrc_left_only":"ncbi-bvbrc",
                "ncbibvbrc_both":"ncbi-bvbrc-gisaid",
                "bvbrc_left_only":"bvbrc",
                "ncbi_left_only":"ncbi",
                "ncbi_both":"ncbi-gisaid",
                "bvbrc_both":"bvbrc-gisaid"
                }

ncbibvbrcgis_mergedmeta["source_merge2"] = ncbibvbrcgis_mergedmeta["merge1_merge2"].astype(str).map(merge_2_dict)

print(ncbibvbrcgis_mergedmeta["source_merge2"].value_counts())

# August/September 2024 pull: All three: 1809, NCBI & GISAID: 453, NCBI & BVBRC: 10927, GISAID: 42347, NCBI: 776, BV-BRC: 1688
# January 2025 pull: All three: 1834, NCBI & GISAID: 486, NCBI & BVBRC: 11531, BVBRC & GISAID: 15, GISAID: 62736, NCBI: 22638, BV-BRC: 1604
# May 2025 pull: All three: 2018, NCBI & GISAID: 686, NCBI & BVBRC: 35490, BVBRC & GISAID: 39, GISAID: 72961, NCBI: 1895, BVBRC: 1732

merge1_merge2
nan_right_only         72961
ncbibvbrc_left_only    35490
ncbibvbrc_both          2018
ncbi_left_only          1895
bvbrc_left_only         1732
ncbi_both                686
bvbrc_both                39
Name: count, dtype: int64
source_merge2
gisaid               72961
ncbi-bvbrc           35490
ncbi-bvbrc-gisaid     2018
ncbi                  1895
bvbrc                 1732
ncbi-gisaid            686
bvbrc-gisaid            39
Name: count, dtype: int64


In [11]:
# Length of merged dataset

print(len(ncbibvbrcgis_mergedmeta))

# August/September 2024 pull: 58,000
# January 2025 pull: 100,844
# May 2025 pull: 114,821

114821


In [12]:
# Add a column for SRA accession

for index, row in ncbibvbrcgis_mergedmeta.iterrows():
    if row["source_merge2"] == "bvbrc" and pd.notna(row["SRA Accession"]):
        ncbibvbrcgis_mergedmeta.at[index, "bvbrc_assembled"] = "yes"
    else:
        ncbibvbrcgis_mergedmeta.at[index, "bvbrc_assembled"] = "no"

ncbibvbrcgis_mergedmeta["bvbrc_assembled"].value_counts()

# August/September 2024 pull: 1288 yes
# January 2025 pull: 1096 yes
# May 2025: 1071 yes

bvbrc_assembled
no     113750
yes      1071
Name: count, dtype: int64

In [13]:
strain_segment_counts = ncbibvbrcgis_mergedmeta["strain_segment"].value_counts()

for value, count in strain_segment_counts.items():
    if count > 1:
        print(value)

In [14]:
# Function: Make a new column that has no "original" and replaces "usda" with "22"

def cleaning_gisaid_id(df, column_name_1, column_name_2):
    # Extract the year from Collection_datetime
    df["year"] = df["Collection_datetime"].str.split("-").str[0]

    # Start with a copy of strain_segment
    df[column_name_2] = df[column_name_1]

    # Remove 'original' from all entries
    df[column_name_2] = df[column_name_2].str.replace("original", "", regex=False)

    # Replace 'usda' with year-specific suffix
    df.loc[(df[column_name_1].str.contains("usda")) & (df["year"] == "2021"), column_name_2] = df[column_name_2].str.replace("usda", "21", regex=False)

    df.loc[(df[column_name_1].str.contains("usda")) & (df["year"] == "2022"), column_name_2] = df[column_name_2].str.replace("usda", "22", regex=False)

    # Optional: warn for unexpected years
    unexpected_years = ~df["year"].isin(["2021", "2022"])
    if df.loc[df[column_name_1].str.contains("usda") & unexpected_years].shape[0] > 0:
        print("Some rows contain 'usda' with unexpected years:")
        print(df.loc[df[column_name_1].str.contains("usda") & unexpected_years, [column_name_1, "year"]])

    # Drop helper column if not needed
    df.drop(columns="year", inplace=True)

    return df

In [15]:
# Remove "original" and replace "usda" with "21" or "22", depending on the year, in the "strain_segment" column
# This allows me to count duplicates in a file that has multiple segments of the same strain

# Check that all sequences with "usda" are from 2022
usda_seqs = ncbibvbrcgis_mergedmeta[ncbibvbrcgis_mergedmeta["strain_segment"].str.contains("usda")]
year_df = pd.DataFrame(usda_seqs["Collection_datetime"].str.split("-", expand = True)[0])
print(year_df.value_counts())

#ncbibvbrcgis_mergedmeta_updated = ncbibvbrcgis_mergedmeta.copy()

ncbibvbrcgis_mergedmeta_updated = cleaning_gisaid_id(ncbibvbrcgis_mergedmeta, "strain_segment", "strain_segment_updated")

0   
2022    10424
2021       16
Name: count, dtype: int64


In [16]:
# Check whether "original" and "usda" are gone
print(ncbibvbrcgis_mergedmeta_updated[ncbibvbrcgis_mergedmeta_updated["strain_segment_updated"].str.contains("original")])
print(ncbibvbrcgis_mergedmeta_updated[ncbibvbrcgis_mergedmeta_updated["strain_segment_updated"].str.contains("usda")])

Empty DataFrame
Columns: [Unnamed: 0_x, Accession, Organism_Name, GenBank_RefSeq, Assembly, SRA_Accession, Release_Date, Length, Nuc_Completeness, Genotype, Segment_x, Geo_Location, Country, USA, Host, Tissue_Specimen_Source, GenBank_Title, Collection_datetime_x, beast_collectdate_x, strain_x, strain_segment, State_x, strain_host_x, order_x, domesticstatus_x, corrected_host_x, flyway_x, state_coord_x, Unnamed: 0_y, Genome ID, NCBI Taxon ID, Species, Genome Status, strain_y, Segment_y, Subtype, H_type, N_type, H5 Clade, Reference, BioProject Accession, SRA Accession, GenBank Accessions, Sequencing Center, Assembly Method, Size, Isolation Source, Collection Date, Collection Year, Isolation Country, Geographic Group, Geographic Location, Host Name, Host Common Name, Host Sex, Host Group, Lab Host, Passage, Additional Metadata, Comments, Date Inserted, Date Modified, Collection_datetime_y, beast_collectdate_y, State_y, strain_host_y, order_y, domesticstatus_y, corrected_host_y, flyway_y, s

In [17]:
ncbibvbrcgis_mergedmeta_updated["strain_segment_updated"].is_unique

False

In [18]:
# ncbibvbrcgis_mergedmeta_updated1 has the "strain_segment_updated" column, where "original" has been removed and "usda" has been replaced with "22"

In [19]:
# Remove 'division' (this is already in the GISAID metadata and it matches the GISAID location)
# Also, drop this first because it will interfere with below

ncbibvbrcgis_mergedmeta_updated1 = ncbibvbrcgis_mergedmeta_updated.drop("division", axis = 1)

In [20]:
# List all the columns so you know what to combine 

print(ncbibvbrcgis_mergedmeta_updated1.columns.values)

['Unnamed: 0_x' 'Accession' 'Organism_Name' 'GenBank_RefSeq' 'Assembly'
 'SRA_Accession' 'Release_Date' 'Length' 'Nuc_Completeness' 'Genotype'
 'Segment_x' 'Geo_Location' 'Country' 'USA' 'Host'
 'Tissue_Specimen_Source' 'GenBank_Title' 'Collection_datetime_x'
 'beast_collectdate_x' 'strain_x' 'strain_segment' 'State_x'
 'strain_host_x' 'order_x' 'domesticstatus_x' 'corrected_host_x'
 'flyway_x' 'state_coord_x' 'Unnamed: 0_y' 'Genome ID' 'NCBI Taxon ID'
 'Species' 'Genome Status' 'strain_y' 'Segment_y' 'Subtype' 'H_type'
 'N_type' 'H5 Clade' 'Reference' 'BioProject Accession' 'SRA Accession'
 'GenBank Accessions' 'Sequencing Center' 'Assembly Method' 'Size'
 'Isolation Source' 'Collection Date' 'Collection Year'
 'Isolation Country' 'Geographic Group' 'Geographic Location' 'Host Name'
 'Host Common Name' 'Host Sex' 'Host Group' 'Lab Host' 'Passage'
 'Additional Metadata' 'Comments' 'Date Inserted' 'Date Modified'
 'Collection_datetime_y' 'beast_collectdate_y' 'State_y' 'strain_host_y'
 

In [21]:
# Columns that need to be combined

# Accession + GenBank Accessions = genbank_acc
# Species_x + Species_y = virus_name
# Length + Size = bp_length
# Segment_x + Segment_y + Segment = segment_no
# Collection_datetime_x + Collection_datetime_y + Collection_datetime = nonambig_date
# beast_collectdate_x + beast_collectdate_y + date = date
# strain_x + strain_y + strain = strain_title
# State_x + State_y + State = division
# strain_host_x + strain_host_y + strain_host = host_fromstrain
# order_x + order_y + order = bird_order
# domesticstatus_x + domesticstatus_y + domesticstatus = domesticstatus_combo
# corrected_host_x + corrected_host_y + corrected_host = hostname_corrected
# flyway_x + flyway_y + flyway = NAflyway
# state_coord_x + state_coord_y + state_coord = state_lat-long
# H5 Clade + h5_clade = ha_clade

In [22]:
# Combine sets of two columns

ncbibvbrcgis_mergedmeta_updated1["genbank_acc"] = ncbibvbrcgis_mergedmeta_updated1["Accession"].combine_first(ncbibvbrcgis_mergedmeta_updated1["GenBank Accessions"])
#ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_virus_name"] = ncbibvbrcgis_mergedmeta_updated1["Species_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Species_y"])
ncbibvbrcgis_mergedmeta_updated1["bp_length"] = ncbibvbrcgis_mergedmeta_updated1["Length"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Size"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_segment"] = ncbibvbrcgis_mergedmeta_updated1["Segment_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Segment_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_nonambigdate"] = ncbibvbrcgis_mergedmeta_updated1["Collection_datetime_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Collection_datetime_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_date"] = ncbibvbrcgis_mergedmeta_updated1["beast_collectdate_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["beast_collectdate_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_strain"] = ncbibvbrcgis_mergedmeta_updated1["strain_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["strain_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_division"] = ncbibvbrcgis_mergedmeta_updated1["State_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["State_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_strain_host"] = ncbibvbrcgis_mergedmeta_updated1["strain_host_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["strain_host_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_order"] = ncbibvbrcgis_mergedmeta_updated1["order_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["order_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_domesticstatus"] = ncbibvbrcgis_mergedmeta_updated1["domesticstatus_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["domesticstatus_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_corrected_host"] = ncbibvbrcgis_mergedmeta_updated1["corrected_host_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["corrected_host_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_flyway"] = ncbibvbrcgis_mergedmeta_updated1["flyway_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["flyway_y"])
ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_statecoord"] = ncbibvbrcgis_mergedmeta_updated1["state_coord_x"].combine_first(ncbibvbrcgis_mergedmeta_updated1["state_coord_y"])

In [23]:
# Combine the third column into sets of two that need a third

ncbibvbrcgis_mergedmeta_updated1["segment_no"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_segment"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Segment"])
ncbibvbrcgis_mergedmeta_updated1["nonambig_date"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_nonambigdate"].combine_first(ncbibvbrcgis_mergedmeta_updated1["Collection_datetime"])
ncbibvbrcgis_mergedmeta_updated1["date_1"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_date"].combine_first(ncbibvbrcgis_mergedmeta_updated1["date"])
# a little worried about these date mergers
ncbibvbrcgis_mergedmeta_updated1["strain_title"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_strain"].combine_first(ncbibvbrcgis_mergedmeta_updated1["strain"])
ncbibvbrcgis_mergedmeta_updated1["division"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_division"].combine_first(ncbibvbrcgis_mergedmeta_updated1["State"])
ncbibvbrcgis_mergedmeta_updated1["host_fromstrain"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_strain_host"].combine_first(ncbibvbrcgis_mergedmeta_updated1["strain_host"])
ncbibvbrcgis_mergedmeta_updated1["bird_order"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_order"].combine_first(ncbibvbrcgis_mergedmeta_updated1["order"])
ncbibvbrcgis_mergedmeta_updated1["domesticstatus_combined"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_domesticstatus"].combine_first(ncbibvbrcgis_mergedmeta_updated1["domesticstatus"])
ncbibvbrcgis_mergedmeta_updated1["host_correction"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_corrected_host"].combine_first(ncbibvbrcgis_mergedmeta_updated1["corrected_host"])
ncbibvbrcgis_mergedmeta_updated1["NAflyway"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_flyway"].combine_first(ncbibvbrcgis_mergedmeta_updated1["flyway"])
ncbibvbrcgis_mergedmeta_updated1["state_lat-long"] = ncbibvbrcgis_mergedmeta_updated1["ncbibvbrc_statecoord"].combine_first(ncbibvbrcgis_mergedmeta_updated1["state_coord"])

In [24]:
# Remove "original" and replace "usda" with "22" in the "strain" column

# Check that all sequences with "usda" are from 2022
usda_seqs_2 = ncbibvbrcgis_mergedmeta_updated1[ncbibvbrcgis_mergedmeta_updated1["strain_segment"].str.contains("usda")]
print(pd.DataFrame(usda_seqs["Collection_datetime"].str.split("-", expand = True)[0].value_counts()))

# Make the same changes: remove "original" and change "usda" to the corresponding year
ncbibvbrcgis_mergedmeta_updated2 = cleaning_gisaid_id(ncbibvbrcgis_mergedmeta_updated1, "strain_title", "strain_title_updated")

      count
0          
2022  10424
2021     16


In [25]:
# Make a column called "strain" to match the FASTA headers for Nextstrain and BEAST

ncbibvbrcgis_mergedmeta_updated2["strain"] = ncbibvbrcgis_mergedmeta_updated2[["strain_title_updated", "date_1", "domesticstatus_combined", "bird_order", 
                                                           "NAflyway"]].astype(str).apply('|'.join, axis=1)

In [26]:
# Add month and year columns and lat and long columns

ncbibvbrcgis_mergedmeta_updated2["year_only"] = ncbibvbrcgis_mergedmeta_updated2["date_1"].str.split("-", expand = True)[0]
ncbibvbrcgis_mergedmeta_updated2["month_only"] = ncbibvbrcgis_mergedmeta_updated2["date_1"].str.split("-", expand = True)[1]

# Don't want lat and long columns because we want to use lat and long from more fine scale goegraphy if we can

# ncbibvbrcgis_mergedmeta_updated2["lat"] = ncbibvbrcgis_mergedmeta_updated2["state_lat-long"].str.split(",", expand = True)[0]
# ncbibvbrcgis_mergedmeta_updated2["long"] = ncbibvbrcgis_mergedmeta_updated2["state_lat-long"].str.split(",", expand = True)[1]

In [27]:
# There are 4 samples (32 sequences) with a 1923 date instead of a 2023 date... I'm going to drop them
# I just can't handle that silliness
# There were 0 samples iwth 1923 date in the May 25, 2025 pull

date_1923 = ncbibvbrcgis_mergedmeta_updated2[ncbibvbrcgis_mergedmeta_updated2["year_only"] == "1923"]

print(len(date_1923))

# date_1923_strains = date_1923["strain_title_updated"].tolist()

# ncbibvbrcgis_mergedmeta_updated3 = ncbibvbrcgis_mergedmeta_updated2[
#     ~ncbibvbrcgis_mergedmeta_updated2["strain_title_updated"].isin(date_1923_strains)]

0


In [28]:
print(len(ncbibvbrcgis_mergedmeta_updated2))

# August/September 2024 pull: 58000
# January 2025 pull: 100,812
# May 25, 2025 pull: 114,821

114821


In [29]:
# ncbibvbrcgis_mergedmeta_updated2 has merged columns and the following additional columns:
# strain_title_updated, year_only, month_only

In [30]:
# Count the number of duplicates and make a dataframe of JUST the duplicates

strain_segment_updated_counts = ncbibvbrcgis_mergedmeta_updated2["strain_segment_updated"].value_counts()

dupes = []

for value, count in strain_segment_updated_counts.items():
    if count > 1:
        dupes.append(value)

print(len(dupes))

# August/September 2024 pull: 9146
# January 2025 pull:9322
# May 25, 2025 pull: 9722

duplicates_remaining_after_merging = ncbibvbrcgis_mergedmeta_updated2[ncbibvbrcgis_mergedmeta_updated2["strain_segment_updated"].isin(dupes)]

9722


In [31]:
# Make a dataframe that is just a list of matching strain names 

columns_to_keep = ["year_only", "strain_title_updated", "strain_title", "date_1", "segment_no", "strain_segment_updated", "source_merge2"]
matching_strains_from_databases = duplicates_remaining_after_merging[columns_to_keep]

In [32]:
# Print the dataframe of duplicates to check for same metadata

duplicates_remaining_after_merging.to_csv("./metadata/sequence_metadata/duplicates_after_merging.csv", sep = ',')

# Also print the limited version with just matching strain names

matching_strains_from_databases.to_csv("./metadata/sequence_metadata/matching_strains_from_databases.csv", sep = ',')

In [33]:
# Deduplicating based on matching strain segments (help from Chat GPT)

def deduplicating_based_on_gisaid_id(df1):

    # First, make a copy of the dataframe
    df2 = df1.copy()

    # Group by the strain_segment_updated column
    groups = df2.groupby('strain_segment_updated')

    rows_to_drop = []

    # Iterate over each group
    for strain_segment, group in groups:
        # Check if more than one source exists
        sources = set(group["source_merge2"])

        # This condition deals with unique strains
        if len(group) == 1:
            pass

        # This condition deals with cases of two strain_segments from the same source, and only keeps the first
        elif len(group) > 1 and len(sources) == 1:
            duplicated_indices = group[group.duplicated(subset="source_merge2", keep='first')].index
            rows_to_drop.extend(duplicated_indices)

        # This condition deals with cases of two strain_segments where at least one row is from GISAID, and removes the GISAID row(s)
        # ex. sources = ["gisaid", "ncbi-bvbrc"] or sources = ["gisaid", "ncbi-bvbrc-gisaid"] or sources = ["gisaid", "gisaid", "ncbi"]
        elif len(sources) > 1 and "gisaid" in sources:
            gisaid_rows = group[group["source_merge2"] == "gisaid"]
            rows_to_drop.extend(gisaid_rows.index)

        # This condition deals with cases of two strain_segments where there is not an option from only GISAID
        # ex. sources = ["ncbi-bvbrc-gisaid", "ncbi-bvbrc"] for a/chicken/michigan/24010308001300/2024_3
        elif len(sources) > 1 and "gisaid" not in sources:
            if "ncbi-bvbrc-gisaid" in sources: 
                alt_rows = group[group["source_merge2"] != "ncbi-bvbrc-gisaid"]
                rows_to_drop.extend(alt_rows.index)
            else:
                print(f"unknown combination for {strain_segment}")
                
        else:
            print(f"unknown combination on {strain_segment}")

    # Drop all rows that need to be dropped
    df2.drop(rows_to_drop, inplace=True)

    return df2

In [34]:
# Test on duplicates_remaining_after_merging dataframe

print(len(duplicates_remaining_after_merging))

duplicated_deduplicated_test = deduplicating_based_on_gisaid_id(duplicates_remaining_after_merging)

print(len(duplicated_deduplicated_test))

# August/September 2024 pull: 20228 duplicates > 9146 deduplicated
# January 2025 pull: 20580 duplicates > 9322 deduplicated
# May 2025 pull: 21380 duplicates > 9722 deduplicated

21380
9722


In [35]:
ncbibvbrcgis_mergedmeta_updated4 = deduplicating_based_on_gisaid_id(ncbibvbrcgis_mergedmeta_updated2)

In [36]:
# Count the length of the new dataset
# This should be (total original length) - (number of duplicated strains) 

print(len(ncbibvbrcgis_mergedmeta_updated4))

# August/September 2024 pull: 46918
# January 2025 pull: 89554
# May 25, 2025 pull: 103,163

103163


In [37]:
# ncbibvbrcgis_mergedmeta_updated3 has dropped duplicates based on strain_segment

print(ncbibvbrcgis_mergedmeta_updated4["strain_segment_updated"].is_unique)

True


In [38]:
columns = ['h5_clade', 'H_type', 'N_type', 'H5 Clade', 'Genotype', 'subtype', 'domesticstatus_combined']
full_data_subtypes = ncbibvbrcgis_mergedmeta_updated4.loc[0:, columns]

full_data_subtypes.to_csv("./metadata/sequence_metadata/full_data_subtypes.tsv", sep = '\t')

In [39]:
# Print ALL columns so you know what to drop

print(ncbibvbrcgis_mergedmeta_updated4.columns.values)

['Unnamed: 0_x' 'Accession' 'Organism_Name' 'GenBank_RefSeq' 'Assembly'
 'SRA_Accession' 'Release_Date' 'Length' 'Nuc_Completeness' 'Genotype'
 'Segment_x' 'Geo_Location' 'Country' 'USA' 'Host'
 'Tissue_Specimen_Source' 'GenBank_Title' 'Collection_datetime_x'
 'beast_collectdate_x' 'strain_x' 'strain_segment' 'State_x'
 'strain_host_x' 'order_x' 'domesticstatus_x' 'corrected_host_x'
 'flyway_x' 'state_coord_x' 'Unnamed: 0_y' 'Genome ID' 'NCBI Taxon ID'
 'Species' 'Genome Status' 'strain_y' 'Segment_y' 'Subtype' 'H_type'
 'N_type' 'H5 Clade' 'Reference' 'BioProject Accession' 'SRA Accession'
 'GenBank Accessions' 'Sequencing Center' 'Assembly Method' 'Size'
 'Isolation Source' 'Collection Date' 'Collection Year'
 'Isolation Country' 'Geographic Group' 'Geographic Location' 'Host Name'
 'Host Common Name' 'Host Sex' 'Host Group' 'Lab Host' 'Passage'
 'Additional Metadata' 'Comments' 'Date Inserted' 'Date Modified'
 'Collection_datetime_y' 'beast_collectdate_y' 'State_y' 'strain_host_y'
 

In [41]:
drop_columns = ['Unnamed: 0_x', 'Accession', 'Organism_Name', 'GenBank_RefSeq', 'Assembly', 'Release_Date', 
                #'Isolate', 'Species_x', 'ncbibvbrc_virus_name','Species_y', 'BioProject',
                'Length', 'Nuc_Completeness', 'Genotype','Segment_x', 'Geo_Location', 'Country', 'USA', 'Host', 'Tissue_Specimen_Source', 
                'GenBank_Title', 'Collection_datetime_x', 'beast_collectdate_x','strain_x', 'State_x', 'strain_host_x', 'order_x', 
                'domesticstatus_x', 'corrected_host_x', 'flyway_x', 'state_coord_x', 'Unnamed: 0_y','NCBI Taxon ID', 'Species', 'Genome Status', 
                'strain_y', 'Segment_y', 'Subtype', 'H_type', 'N_type', 'Reference', 'BioProject Accession','GenBank Accessions', 'Sequencing Center', 
                'Assembly Method', 'Size', 'Isolation Source', 'Collection Date', 'Collection Year', 'Isolation Country','Geographic Group', 
                'Geographic Location', 'Host Name', 'Host Common Name', 'Host Sex', 'Host Group', 'Lab Host', 'Passage', 'Additional Metadata', 'Comments', 
                'Date Inserted','Date Modified', 'Collection_datetime_y', 'beast_collectdate_y', 'State_y', 'strain_host_y', 'order_y', 'domesticstatus_y', 
                'corrected_host_y', 'flyway_y', 'state_coord_y', '_merge_1', 'source', 'Unnamed: 0', 'virus', 'isolate_id', 'date', 'region', 'country', 
                'location', 'host', 'domestic_status', 'subtype','originating_lab', 'submitting_lab', 'authors', 'PMID', 'gisaid_clade', 'n_segments', 
                'genoflu', 'genoflu_PB2', 'genoflu_PB1', 'genoflu_PA', 'genoflu_HA', 'genoflu_NP', 'genoflu_NA', 'genoflu_MP', 'genoflu_NS', 'State', 
                'strain_host', 'order', 'domesticstatus', 'corrected_host','flyway', 'state_coord', 'Segment', 'Collection_datetime', '_merge_2', 
                'merge1_merge2', 'ncbibvbrc_segment', 'ncbibvbrc_nonambigdate', 'ncbibvbrc_date', 'ncbibvbrc_strain', 'ncbibvbrc_division', 
                'ncbibvbrc_strain_host', 'ncbibvbrc_order', 'ncbibvbrc_domesticstatus', 'ncbibvbrc_corrected_host', 'ncbibvbrc_flyway', 'ncbibvbrc_statecoord']

ncbibvbrcgis_final = ncbibvbrcgis_mergedmeta_updated4.drop(labels=drop_columns, axis = 1)

In [42]:
# Rename the date_1 column to date
# This is necessary for Nextstrain
# And you couldn't do it before because you needed the column originally named date, which you have now deleted

ncbibvbrcgis_final = ncbibvbrcgis_final.rename({"date_1":"date"}, axis = 1)

In [43]:
# Save this final as a TSV

#ncbibvbrcgis_final.to_csv("./metadata/sequence_metadata/BVBRC+NCBI+GIS_FINAL.tsv", sep = '\t')
ncbibvbrcgis_final.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_FINAL.tsv", sep = '\t')

In [44]:
print(len(ncbibvbrcgis_final))

# August/September 2024 pull: 46918
# January 2025 pull: 89554
# May 25, 2025 pull: 103,163

103163


In [54]:
# Split into eight metadata files for each segment

ncbibvbrcgis_pb2 = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 1.0] # Encodes PB2
ncbibvbrcgis_pb1 = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 2.0] # Encodes PB1
ncbibvbrcgis_pa = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 3.0] # Encodes PA
ncbibvbrcgis_ha = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 4.0] # Encodes HA
ncbibvbrcgis_np = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 5.0] # Encodes NP
ncbibvbrcgis_na = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 6.0] # Encodes NA
ncbibvbrcgis_m = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 7.0] # Encodes M1
ncbibvbrcgis_ns = ncbibvbrcgis_final[ncbibvbrcgis_final["segment_no"] == 8.0] # Encodes NS

In [55]:
print(ncbibvbrcgis_pb2["strain"].is_unique)
print(ncbibvbrcgis_pb1["strain"].is_unique)
print(ncbibvbrcgis_pa["strain"].is_unique)
print(ncbibvbrcgis_ha["strain"].is_unique)
print(ncbibvbrcgis_np["strain"].is_unique)
print(ncbibvbrcgis_na["strain"].is_unique)
print(ncbibvbrcgis_m["strain"].is_unique)
print(ncbibvbrcgis_ns["strain"].is_unique)

True
True
True
True
True
True
True
True


In [56]:
# There is another duplication error that needs attention: 
# Some strains have the same numerical code, but one says "ak" in the strain title and the other says "alaska"

In [57]:
# Function: makes a dataset of duplicates where the strain_code, division, date, and host were all the same
def strain_code_duplicates(df):

    df_2 = df.copy()

    df_2["strain_code"] = (df_2["strain_title_updated"].str.split("/", expand = True)[3].astype(str))
    
    duplicates_df = []
    
    # Group by all three fields
    for (code, location, date, host), group in df_2.groupby(["strain_code", "division", "nonambig_date", "host_fromstrain"]):
        if len(group) > 1:  # only collect actual duplicates
            # print(group["source_merge2"].unique())
            source = set(group["source_merge2"])
            # print(source)
            if source == {"gisaid", "ncbi-bvbrc"}: # If the duplicates are from ncbi-bvbrc and gisaid, keep ncbi-bvbrc
                duplicates_df.append(group[group["source_merge2"] == "gisaid"])
            elif source == {'gisaid', 'ncbi'}: # If the duplicates are from ncbi and gisaid, keep ncbi
                duplicates_df.append(group[group["source_merge2"] == "gisaid"])
            elif source == {'gisaid', 'bvbrc'}: # If the duplicates are from bvbrc and gisaid, keep bvbrc
                duplicates_df.append(group[group["source_merge2"] == "gisaid"])
            elif source == {'ncbi-bvbrc-gisaid', 'ncbi-bvbrc'}: # If the duplicates are from ncbi-bvbrc-gisaid and ncbi-bvbrc, keep ncbi-bvbrc
                duplicates_df.append(group[group["source_merge2"] == "ncbi-bvbrc-gisaid"])
            elif source == {'ncbi-gisaid', 'ncbi'}: # If the duplicates are from ncbi-gisaid and ncbi, keep ncbi
                duplicates_df.append(group[group["source_merge2"] == "ncbi-gisaid"])
            elif len(source) == 1:
                duplicates_df.append(group.head(1))
            else:
                print(f"Something unexpected happened with strain_code {code}")
        
    
    # Combine all matching groups into a single DataFrame
    duplicates_df_concat = pd.concat(duplicates_df, ignore_index=True)

    # Make a list of strains
    strain_list = duplicates_df_concat["strain"].to_list()

    return(duplicates_df_concat, strain_list)

In [58]:
pb2_duplicates, pb2_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_pb2)
pb1_duplicates, pb1_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_pb1)
pa_duplicates, pa_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_pa)
ha_duplicates, ha_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_ha)
np_duplicates, np_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_np)
na_duplicates, na_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_na)
m_duplicates, m_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_m)
ns_duplicates, ns_duplicate_strains = strain_code_duplicates(ncbibvbrcgis_ns)

In [59]:
# Deduplicate (again)

pb2_deduped = ncbibvbrcgis_pb2[~ncbibvbrcgis_pb2["strain"].isin(pb2_duplicate_strains)]
pb1_deduped = ncbibvbrcgis_pb1[~ncbibvbrcgis_pb1["strain"].isin(pb1_duplicate_strains)]
pa_deduped = ncbibvbrcgis_pa[~ncbibvbrcgis_pa["strain"].isin(pa_duplicate_strains)]
ha_deduped = ncbibvbrcgis_ha[~ncbibvbrcgis_ha["strain"].isin(ha_duplicate_strains)]
np_deduped = ncbibvbrcgis_np[~ncbibvbrcgis_np["strain"].isin(np_duplicate_strains)]
na_deduped = ncbibvbrcgis_na[~ncbibvbrcgis_na["strain"].isin(na_duplicate_strains)]
m_deduped = ncbibvbrcgis_m[~ncbibvbrcgis_m["strain"].isin(m_duplicate_strains)]
ns_deduped = ncbibvbrcgis_ns[~ncbibvbrcgis_ns["strain"].isin(ns_duplicate_strains)]

In [61]:
print(len(pb2_deduped))
print(len(pb1_deduped))
print(len(pa_deduped))
print(len(ha_deduped))
print(len(np_deduped))
print(len(na_deduped))
print(len(m_deduped))
print(len(ns_deduped))

# August/September 2024 pull: Adds up to 46,981 (in order: 5770, 5771, 5820, 5927, 5906, 5919, 5899, 5906
# January 2025 pull: Adds up to 89,554 (in order: 11099, 11099, 11149, 11255, 11235, 11244, 11228, 11245)
# May 25, 2025 pull: Adds up to 103,163 (in order: 12795, 12790, 12849, 12962, 12940, 12945, 12934, 12948)
# May 25, 2025 pull (updated with deduplicated numbers): 10142, 10140, 10193, 10305, 10283, 10294, 10277, 10291

10142
10140
10193
10305
10283
10294
10277
10291


In [62]:
# Save each segment

# ncbibvbrcgis_pb2.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_pb2.tsv", sep = '\t')
# ncbibvbrcgis_pb1.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_pb1.tsv", sep = '\t')
# ncbibvbrcgis_pa.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_pa.tsv", sep = '\t')
# ncbibvbrcgis_ha.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_ha.tsv", sep = '\t')
# ncbibvbrcgis_np.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_np.tsv", sep = '\t')
# ncbibvbrcgis_na.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_na.tsv", sep = '\t')
# ncbibvbrcgis_m.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_m.tsv", sep = '\t')
# ncbibvbrcgis_ns.to_csv("./metadata/sequence_metadata/NCBI+BVBRC+GIS_ns.tsv", sep = '\t')

# ncbibvbrcgis_pb2.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb2.tsv", sep = '\t')
# ncbibvbrcgis_pb1.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb1.tsv", sep = '\t')
# ncbibvbrcgis_pa.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pa.tsv", sep = '\t')
# ncbibvbrcgis_ha.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha.tsv", sep = '\t')
# ncbibvbrcgis_np.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_np.tsv", sep = '\t')
# ncbibvbrcgis_na.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_na.tsv", sep = '\t')
# ncbibvbrcgis_m.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_m.tsv", sep = '\t')
# ncbibvbrcgis_ns.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ns.tsv", sep = '\t')

# ncbibvbrcgis_pb2.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb2.tsv", sep = '\t')
# ncbibvbrcgis_pb1.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb1.tsv", sep = '\t')
# ncbibvbrcgis_pa.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pa.tsv", sep = '\t')
# ncbibvbrcgis_ha.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha.tsv", sep = '\t')
# ncbibvbrcgis_np.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_np.tsv", sep = '\t')
# ncbibvbrcgis_na.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_na.tsv", sep = '\t')
# ncbibvbrcgis_m.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_m.tsv", sep = '\t')
# ncbibvbrcgis_ns.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ns.tsv", sep = '\t')

pb2_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb2.tsv", sep = '\t')
pb1_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pb1.tsv", sep = '\t')
pa_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_pa.tsv", sep = '\t')
ha_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ha.tsv", sep = '\t')
np_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_np.tsv", sep = '\t')
na_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_na.tsv", sep = '\t')
m_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_m.tsv", sep = '\t')
ns_deduped.to_csv("./metadata/sequence_metadata/NCBI-BVBRC-GIS_ns.tsv", sep = '\t')

In [ ]:
# # Deduplicated based on matching strain_segment

# # If strain_segment_updated matches another strain_segment_updated
# # Then, keep the ncbi-bvbrc segment and remove the gisaid segment

# def deduplicating_based_on_gisaid_id(df1):

#     # First, make a copy of the dataframe in question
#     df2 = df1.copy()

#     # Next, find rows that match each strain_segment combination
#     for idx, row in df2.iterrows():
#         strain_segment = df2["strain_segment_updated"]
#         matching_rows = df2[df2["strain_segment_updated"] == strain_segment]

#         # If there are no matching rows for that row, pass
#         if matching_rows.empty:
#             continue

#         # If there are matching rows and one is from ncbi-bvbrc and one is from gisaid, drop the ncbi-bvbrc row
#         if set(matching_rows["source_merge2"]) == {"ncbi-bvbrc", "gisaid"}:
#             # Find the row from "ncbi-bvbrc" and keep it
#             gisaid_row = matching_rows[matching_rows["source_merge2"] == "gisaid"]
#             df2.drop(gisaid_row.index, inplace=True)

#         elif set(matching_rows["source_merge2"]) == {"ncbi", "gisaid"}: 
#             # Find the row from "ncbi" and keep it
#             gisaid_row = matching_rows[matching_rows["source_merge2"] == "gisaid"]
#             df2.drop(gisaid_row.index, inplace=True)

#         elif set(matching_rows["source_merge2"]) == {"bvbrc", "gisaid"}:
#             # Find the row from "bvbrc" and keep it
#             gisaid_row = matching_rows[matching_rows["source_merge2"] == "gisaid"]
#             df2.drop(gisaid_row.index, inplace=True)

#         elif set(matching_rows["source_merge2"]) == {"ncbi-bvbrc-gisaid", "gisaid"}:
#             # Find the row from "ncbi-bvbrc-gisaid" and keep it
#             gisaid_row = matching_rows[matching_rows["source_merge2"] == "gisaid"]
#             df2.drop(gisaid_row.index, inplace=True)

#         # If there are matching rows with some other situation, print "something else happened"
#         else:
#             print(f"Unexpected situation for strain_segment: {strain_segment}")

#     return(df2)